**Please**: Read this https://publish.obsidian.md/minceddata/Getting+social/Blog/Monitoring+Fabric/Entra+ID+Group+Expansion+-+the+complete+guide before you run this notebook in a production environment

I did, and I'm happy with the results ;-)

However, it will consume Compute Unit (s)

If you are going to use the result for decision making (I do), make sure that the results are correct.

The last run in a production environment before publishing this notebook to my git repository finished 
+ after 1 hour and 25 minutes with detail and summary logging enabled
+ after 23 minutes only with summary logging enabled

It expanded –5k groups into 71k rows of expanded users with ~20k distinct users. This means there are 20k people who are able to access one or more morkspaces.

Please be aware that I'm talking about workspace users not app users.

# Sec 01 - Configuraton and Imports

## Configuration

In [ ]:
# =============================================================================
# CONFIGURATION:
# =============================================================================
# Toggle CSV logging options here
ENABLE_DETAIL_CSV = False  # Set to True for detailed CSV logs (easy to read in Fabric!)
ENABLE_SUMMARY_CSV = True  # Set to False to disable summary CSV

# =============================================================================
# The NAME(s) of the delta table(s) that will be created in the default lakehouse
# If the table does not exist it will be created automatically, ff you run this notebook the first time. 
# Subsequent runs will use this table.
# Please keep in mind that data will be written in batches.
# The first batch will use the OVERWRITE mode, subsequent writes will use APPEND.
# There is no tracking of history, this should be done by other tools ;-)
# Choose a reasonable name.
# The fancy prefix "cst_" is there to make the tables distinct from the FUAM tables. I use the FUAM_Lakehouse
# as the default lakehouse.
TABLE_NAME = "cst_entra_id_group_memberships"
WRITE_DELTA_TABLE_BATCH_SIZE = 10000

# The table name that will be used for Row Level Security in a semantic models
# basically the reason why I'm doing this.
TABLE_NAME_FOR_RLS = "cst_workspaces_scanned_users_expandedgroupsforrls"

# =============================================================================
# Authentication:
# The Azure Key Vault 
key_vault = "<here goes the link to your Azure Key Vault>"

# The tenant id 
tenant_id = "<here goes your tenant id>"

# The Application Id (Client Id) of the service principal account 
client_id = "<here goes the client id of the service principal>"

# Name of the secret, this is of course not the secret ;-)
name_of_the_spn_secret = "<the name of the secret>"
# Fetching the Client Secret for the service principal account with permissions on the GraphAPI
# Of course everyone who is executing the notebook needs to be allowed to read the secret ;-)
client_secret = mssparkutils.credentials.getSecret(key_vault , f"{name_of_the_spn_secret}")

authority_url = f"https://login.microsoftonline.com/{tenant_id}"

## Imports

In [ ]:
# =============================================================================
# Imports:
# =============================================================================
# standard libraries
from typing import Dict, Set, List, Tuple, Optional, Callable
import time
from datetime import datetime
import logging
import os
import csv
from collections import deque

import typing

# 3rd party libraries
import requests
import msal
from pyspark.sql.functions import when

# Sec 02 - Classes

## Class - FabricLoggerCSV

In [ ]:
class FabricLoggerCSV:
    """
    CSV-based logger for Microsoft Fabric notebooks.
    - Summary CSV: One row with run metrics (always created)
    - Detail CSV: Multiple rows with timestamped log entries (optional)
    Both CSVs are easily viewable and queryable in Fabric UI!
    """
    
    def __init__(
        self, 
        log_name: str, 
        lakehouse_path: str = "/lakehouse/default/Files/logs", 
        enable_detail_csv: bool = False,  # Default: OFF
        enable_summary_csv: bool = True    # Default: ON
    ):
        """
        Initialize CSV logger with configurable outputs
        
        Args:
            log_name: Name prefix for log files
            lakehouse_path: Path to logs directory
            enable_detail_csv: If True, writes detailed logs to CSV (default: False)
            enable_summary_csv: If True, writes run summary to CSV (default: True)
        """
        self.log_name = log_name
        self.lakehouse_path = lakehouse_path
        self.enable_detail_csv = enable_detail_csv
        self.enable_summary_csv = enable_summary_csv
        
        # Create logs directory if it doesn't exist
        os.makedirs(lakehouse_path, exist_ok=True)
        
        # Create timestamp
        self.timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.run_start = datetime.now()
        
        # Setup detail CSV (if enabled)
        if self.enable_detail_csv:
            self.detail_filename = f"{log_name}_detail_{self.timestamp}.csv"
            self.detail_filepath = os.path.join(lakehouse_path, self.detail_filename)
            
            # Create detail CSV file with headers
            with open(self.detail_filepath, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['timestamp', 'level', 'message'])
            
            self._log_detail('INFO', f'=== Log started: {log_name} ===')
            self._log_detail('INFO', f'Run ID: {self.timestamp}')
            self._log_detail('INFO', f'Detail log: {self.detail_filepath}')
        else:
            self.detail_filepath = None
        
        # Setup summary CSV data (if enabled)
        if self.enable_summary_csv:
            self.summary_filename = f"{log_name}_summary_{self.timestamp}.csv"
            self.summary_filepath = os.path.join(lakehouse_path, self.summary_filename)
            self.summary_data = {
                'run_id': self.timestamp,
                'start_time': self.run_start.isoformat(),
                'detail_log': self.detail_filename if self.enable_detail_csv else 'N/A'
            }
            if self.enable_detail_csv:
                self._log_detail('INFO', f'Summary CSV: {self.summary_filepath}')
        else:
            self.summary_filepath = None
            self.summary_data = None
    
    def _log_detail(self, level: str, message: str):
        """Write a log entry to the detail CSV"""
        if self.enable_detail_csv:
            timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]  # milliseconds
            with open(self.detail_filepath, 'a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([timestamp, level, message])
    
    def info(self, message: str):
        """Log an info message to detail CSV (if enabled)"""
        self._log_detail('INFO', message)
    
    def warning(self, message: str):
        """Log a warning message to detail CSV (if enabled)"""
        self._log_detail('WARNING', message)
    
    def error(self, message: str):
        """Log an error message to detail CSV (if enabled)"""
        self._log_detail('ERROR', message)
    
    def debug(self, message: str):
        """Log a debug message to detail CSV (if enabled)"""
        self._log_detail('DEBUG', message)
    
    def set_summary_metric(self, key: str, value):
        """Set a metric for the summary CSV (only if enabled)"""
        if self.enable_summary_csv and self.summary_data is not None:
            self.summary_data[key] = value
    
    def close(self):
        """Close the logger and write summary CSV"""
        run_end = datetime.now()
        duration = (run_end - self.run_start).total_seconds()
        
        # Log to detail CSV (if enabled)
        if self.enable_detail_csv:
            self._log_detail('INFO', f'=== Log ended: {self.log_name} ===')
            self._log_detail('INFO', f'Total duration: {duration:.2f} seconds')
        
        # Write summary CSV (if enabled)
        if self.enable_summary_csv and self.summary_data is not None:
            self.summary_data['end_time'] = run_end.isoformat()
            self.summary_data['duration_seconds'] = round(duration, 2)
            
            # Write summary CSV
            with open(self.summary_filepath, 'w', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=self.summary_data.keys())
                writer.writeheader()
                writer.writerow(self.summary_data)
            
            if self.enable_detail_csv:
                self._log_detail('INFO', f'✓ Summary CSV written to: {self.summary_filepath}')
    
    def get_detail_path(self) -> str:
        """Get the full path to the detail CSV (returns None if disabled)"""
        return self.detail_filepath
    
    def get_summary_path(self) -> str:
        """Get the full path to the summary CSV (returns None if disabled)"""
        return self.summary_filepath
    
    def __enter__(self):
        """Context manager entry"""
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        """Context manager exit - ensures log is properly closed"""
        if exc_type is not None:
            if self.enable_detail_csv:
                self._log_detail('ERROR', f'Exception occurred: {exc_type.__name__}: {exc_val}')
            if self.enable_summary_csv and self.summary_data is not None:
                self.summary_data['status'] = 'failed'
                self.summary_data['error'] = str(exc_val)
        else:
            if self.enable_summary_csv and self.summary_data is not None:
                self.summary_data['status'] = 'success'
        self.close()

print("✓ FabricLoggerCSV class loaded")
print("  - enable_detail_csv: Detailed logs as CSV (default: False)")
print("  - enable_summary_csv: Summary metrics as CSV (default: True)")
print("  - Both CSVs viewable in Fabric UI!")

## Class - GroupMetadataFetcher

In [ ]:
class GroupMetadataFetcher:
    """
    Fetches metadata (displayName, description, etc.) for Entra ID groups.
    Companion class to EntraIDGroupExpander - follows same design patterns.
    
    Features:
    - JSON BATCHING: Batch up to 20 API requests into a single HTTP call
    - TOKEN REFRESH: Automatic token refresh on 401 errors
    - RETRY LOGIC: Exponential backoff for rate limiting and transient errors
    - STATISTICS: Track API calls, cache hits, and performance metrics
    - CACHING: Optional in-memory cache to avoid duplicate API calls
    
    Usage:
        fetcher = GroupMetadataFetcher(
            access_token=token,
            token_refresh_callback=get_fresh_token,
            use_cache=True
        )
        
        # Fetch displayNames for multiple groups
        display_names = fetcher.fetch_display_names(group_ids_list)
        
        # Fetch full metadata (displayName, description, mail, etc.)
        metadata = fetcher.fetch_metadata(group_ids_list, fields=['displayName', 'description'])
    """
    
    def __init__(
        self,
        access_token: str,
        token_refresh_callback: Optional[Callable[[], str]] = None,
        max_retries: int = 3,
        base_delay: float = 1.0,
        max_delay: float = 60.0,
        batch_size: int = 20,
        batch_delay: float = 0.2,
        use_cache: bool = True
    ):
        """
        Initialize GroupMetadataFetcher.
        
        Args:
            access_token: Microsoft Graph API access token
            token_refresh_callback: Optional callback function to refresh expired tokens
            max_retries: Maximum number of retry attempts for failed requests
            base_delay: Base delay in seconds for exponential backoff
            max_delay: Maximum delay in seconds between retries
            batch_size: Number of requests per batch (max 20)
            batch_delay: Delay between batch requests to avoid rate limiting
            use_cache: Whether to cache fetched metadata
        """
        self.access_token = access_token
        self.token_refresh_callback = token_refresh_callback
        self.max_retries = max_retries
        self.base_delay = base_delay
        self.max_delay = max_delay
        self.batch_size = min(batch_size, 20)  # Graph API limit
        self.batch_delay = batch_delay
        self.use_cache = use_cache
        
        # Cache for fetched metadata
        self.cache: Dict[str, Dict] = {}
        
        # Statistics tracking
        self.stats = {
            'api_calls': 0,
            'batch_requests': 0,
            'batched_api_calls': 0,
            'successful_fetches': 0,
            'fallback_to_id': 0,
            'cache_hits': 0,
            'token_refreshes': 0,
            'rate_limit_429': 0,
            'errors_404': 0,
            'errors_403': 0,
            'errors_other': 0
        }
    
    def _get_headers(self) -> Dict[str, str]:
        """Get request headers with current access token"""
        return {
            'Authorization': f'Bearer {self.access_token}',
            'Content-Type': 'application/json',
            'ConsistencyLevel': 'eventual'
        }
    
    def _refresh_token(self, logger=None) -> bool:
        """Refresh the access token using the callback function"""
        if self.token_refresh_callback is None:
            return False
        
        try:
            new_token = self.token_refresh_callback()
            if new_token:
                self.access_token = new_token
                self.stats['token_refreshes'] += 1
                if logger:
                    logger.info("✓ Token refreshed successfully")
                return True
        except Exception as e:
            if logger:
                logger.error(f"⚠️ Token refresh failed: {e}")
        
        return False
    
    def _make_batch_request(
        self,
        group_ids: List[str],
        select_fields: List[str],
        logger=None
    ) -> Dict[str, Optional[Dict]]:
        """
        Make a JSON batch request to fetch metadata for multiple groups.
        
        Args:
            group_ids: List of group IDs to fetch
            select_fields: List of fields to retrieve (e.g., ['displayName', 'description'])
            logger: Optional logger instance
            
        Returns:
            Dictionary mapping group_id to metadata dict (or None if fetch failed)
        """
        if len(group_ids) > self.batch_size:
            if logger:
                logger.warning(f"Batch size {len(group_ids)} exceeds limit of {self.batch_size}, truncating")
            group_ids = group_ids[:self.batch_size]
        
        # Build select parameter
        select_param = ','.join(select_fields) if select_fields else 'displayName'
        
        # Prepare batch requests
        batch_requests = [
            {
                "id": str(idx),
                "method": "GET",
                "url": f"/groups/{group_id}?$select={select_param}"
            }
            for idx, group_id in enumerate(group_ids)
        ]
        
        batch_payload = {"requests": batch_requests}
        
        self.stats['batch_requests'] += 1
        self.stats['batched_api_calls'] += len(group_ids)
        
        if logger:
            logger.debug(f"Making batch request for {len(group_ids)} groups")
        
        retry_count = 0
        while retry_count <= self.max_retries:
            try:
                response = requests.post(
                    "https://graph.microsoft.com/v1.0/$batch",
                    headers=self._get_headers(),
                    json=batch_payload,
                    timeout=60
                )
                
                if response.status_code == 200:
                    batch_response = response.json()
                    results = {}
                    
                    for response_item in batch_response.get('responses', []):
                        request_id = int(response_item.get('id'))
                        group_id = group_ids[request_id]
                        status = response_item.get('status')
                        
                        if status == 200:
                            body = response_item.get('body', {})
                            results[group_id] = body
                            self.stats['successful_fetches'] += 1
                            
                            if logger:
                                display_name = body.get('displayName', 'N/A')
                                logger.debug(f"  ✓ {group_id}: {display_name}")
                        
                        elif status == 404:
                            self.stats['errors_404'] += 1
                            if logger:
                                logger.warning(f"  ✗ Group not found (404): {group_id}")
                            results[group_id] = None
                        
                        elif status == 403:
                            self.stats['errors_403'] += 1
                            if logger:
                                logger.warning(f"  ✗ Access denied (403): {group_id}")
                            results[group_id] = None
                        
                        else:
                            self.stats['errors_other'] += 1
                            if logger:
                                logger.warning(f"  ✗ Error {status} for group: {group_id}")
                            results[group_id] = None
                    
                    return results
                
                elif response.status_code == 401:
                    if logger:
                        logger.warning(f"Token expired (401), attempting refresh...")
                    if self._refresh_token(logger):
                        continue  # Retry with new token
                    else:
                        if logger:
                            logger.error("Failed to refresh token")
                        return {}
                
                elif response.status_code == 429:
                    self.stats['rate_limit_429'] += 1
                    retry_after = int(response.headers.get('Retry-After', self.base_delay * (2 ** retry_count)))
                    wait_time = min(retry_after, self.max_delay)
                    
                    if logger:
                        logger.warning(f"Rate limited (429), waiting {wait_time}s before retry {retry_count + 1}/{self.max_retries}")
                    
                    time.sleep(wait_time)
                    retry_count += 1
                
                else:
                    if logger:
                        logger.error(f"Batch request failed: {response.status_code} - {response.text}")
                    return {}
            
            except requests.exceptions.Timeout:
                wait_time = min(self.base_delay * (2 ** retry_count), self.max_delay)
                if logger:
                    logger.warning(f"Request timeout, waiting {wait_time}s before retry {retry_count + 1}/{self.max_retries}")
                time.sleep(wait_time)
                retry_count += 1
            
            except requests.exceptions.RequestException as e:
                if logger:
                    logger.error(f"Batch request exception: {e}")
                return {}
        
        if logger:
            logger.error(f"Max retries ({self.max_retries}) exceeded for batch request")
        return {}
    
    def fetch_metadata(
        self,
        group_ids: List[str],
        fields: Optional[List[str]] = None,
        logger=None
    ) -> Dict[str, Dict]:
        """
        Fetch metadata for multiple groups using JSON batching.
        
        Args:
            group_ids: List of group IDs to fetch
            fields: Optional list of fields to retrieve. Defaults to ['displayName']
            logger: Optional logger instance
            
        Returns:
            Dictionary mapping group_id to metadata dict
            Format: {group_id: {'displayName': '...', 'description': '...', ...}}
        """
        if fields is None:
            fields = ['displayName']
        
        if logger:
            logger.info(f"Fetching metadata for {len(group_ids)} groups (fields: {', '.join(fields)})")
        
        results = {}
        groups_to_fetch = []
        
        # Check cache first
        for group_id in group_ids:
            if self.use_cache and group_id in self.cache:
                self.stats['cache_hits'] += 1
                results[group_id] = self.cache[group_id]
                if logger:
                    logger.debug(f"Cache hit for group: {group_id}")
            else:
                groups_to_fetch.append(group_id)
        
        if logger and self.use_cache:
            logger.info(f"Cache hits: {len(results)}, Fetching: {len(groups_to_fetch)}")
        
        # Fetch remaining groups in batches
        for i in range(0, len(groups_to_fetch), self.batch_size):
            batch = groups_to_fetch[i:i + self.batch_size]
            
            batch_results = self._make_batch_request(batch, fields, logger)
            
            for group_id, metadata in batch_results.items():
                if metadata is not None:
                    if self.use_cache:
                        self.cache[group_id] = metadata
                    results[group_id] = metadata
                else:
                    # Fallback: create minimal metadata with just the ID
                    self.stats['fallback_to_id'] += 1
                    fallback_metadata = {'id': group_id}
                    for field in fields:
                        fallback_metadata[field] = group_id
                    results[group_id] = fallback_metadata
            
            # Delay between batches to avoid rate limiting
            if i + self.batch_size < len(groups_to_fetch):
                time.sleep(self.batch_delay)
        
        if logger:
            logger.info(f"✓ Fetched metadata for {len(results)} groups")
        
        return results
    
    def fetch_display_names(
        self,
        group_ids: List[str],
        logger=None
    ) -> Dict[str, str]:
        """
        Convenience method to fetch only displayNames for groups.
        
        Args:
            group_ids: List of group IDs to fetch
            logger: Optional logger instance
            
        Returns:
            Dictionary mapping group_id to displayName (falls back to group_id if fetch fails)
        """
        metadata = self.fetch_metadata(group_ids, fields=['displayName'], logger=logger)
        
        # Extract displayNames, fallback to group_id if not found
        display_names = {
            group_id: meta.get('displayName', group_id)
            for group_id, meta in metadata.items()
        }
        
        return display_names
    
    def clear_cache(self):
        """Clear the metadata cache"""
        self.cache.clear()
    
    def get_stats(self) -> Dict[str, int]:
        """Get statistics about API calls and performance"""
        return self.stats.copy()
    
    def print_stats(self, logger=None):
        """Print statistics summary"""
        output = [
            f"\n{'='*70}",
            f"GROUP METADATA FETCHER STATISTICS",
            f"{'='*70}",
            f"API Statistics:",
            f"  Batch requests: {self.stats['batch_requests']}",
            f"  Batched API calls: {self.stats['batched_api_calls']}",
            f"  Successful fetches: {self.stats['successful_fetches']}",
            f"  Fallback to ID: {self.stats['fallback_to_id']}",
            f"",
            f"Cache Statistics:",
            f"  Cache hits: {self.stats['cache_hits']}",
            f"  Items in cache: {len(self.cache)}",
            f"",
            f"Error Statistics:",
            f"  Token refreshes: {self.stats['token_refreshes']}",
            f"  Rate limit hits (429): {self.stats['rate_limit_429']}",
            f"  Not found (404): {self.stats['errors_404']}",
            f"  Access denied (403): {self.stats['errors_403']}",
            f"  Other errors: {self.stats['errors_other']}",
            f"{'='*70}\n"
        ]
        
        message = '\n'.join(output)
        
        if logger:
            for line in output:
                logger.info(line)
        else:
            print(message)


print("✓ GroupMetadataFetcher class loaded")
print("  Key features:")
print("  - JSON batching (up to 20 groups per request)")
print("  - Automatic token refresh and retry logic")
print("  - In-memory caching to avoid duplicate API calls")
print("  - Detailed statistics tracking")
print("  - Flexible metadata field selection")
print("\n  Usage:")
print("    fetcher = GroupMetadataFetcher(token, token_refresh_callback=get_fresh_token)")
print("    display_names = fetcher.fetch_display_names(group_ids_list)")
print("    metadata = fetcher.fetch_metadata(group_ids_list, fields=['displayName', 'description'])")

## Class - EntraIDGroupExpander (ITERATIVE VERSION)

- Uses while loop with work queue instead
- Manages queue with deque for efficient append/pop operations
- Maintains same caching behavior to avoid duplicate expansions
- Handles groups appearing at multiple nesting levels
- Preserves batched writing mechanism

In [ ]:
class EntraIDGroupExpander:
    """
    Iterative (non-recursive) group expansion using work queue.
    Expands Entra ID (Azure AD) groups with retry logic and automatic token refresh.
    
    Features:
    - LOCAL cache: Cache reset per root group (default)
    - GLOBAL cache: Cache shared across all root groups (prevents duplicate API calls)
    - JSON BATCHING: Batch up to 20 API requests into a single HTTP call
    - PAGINATION: Properly handles pagination in both individual and batch requests
    - TABLE MANAGEMENT: Option to overwrite table on first write (prevents duplicates)
    
    FIXED: 
    - Batch requests properly handle pagination for groups with >100 members
    - Table can be cleared before writing (no more duplicates!)
    """
    
    def __init__(
        self,
        access_token: str,
        token_refresh_callback: Optional[Callable[[], str]] = None,
        max_retries: int = 5,
        base_delay: float = 2.0,
        max_delay: float = 120.0,
        batch_delay: float = 0.2,
        use_global_cache: bool = False,
        use_json_batching: bool = False,
        batch_size: int = 20
    ):
        self.access_token = access_token
        self.token_refresh_callback = token_refresh_callback
        self.max_retries = max_retries
        self.base_delay = base_delay
        self.max_delay = max_delay
        self.batch_delay = batch_delay
        self.use_global_cache = use_global_cache
        self.use_json_batching = use_json_batching
        self.batch_size = min(batch_size, 20)
        
        # GLOBAL cache
        self.global_cache: Dict[str, List[Dict]] = {}
        
        # Statistics tracking
        self.stats = {
            'api_calls': 0,
            'batch_requests': 0,
            'batched_api_calls': 0,
            'pagination_calls': 0,
            'cache_hits': 0,
            'global_cache_hits': 0,
            'local_cache_hits': 0,
            'token_refreshes': 0,
            'rate_limit_429': 0,
            'max_nesting_depth': 0
        }
    
    def _get_headers(self) -> Dict[str, str]:
        """Get request headers with current access token"""
        return {
            'Authorization': f'Bearer {self.access_token}',
            'Content-Type': 'application/json',
            'ConsistencyLevel': 'eventual'
        }
    
    def _refresh_token(self) -> bool:
        """Refresh the access token using the callback function"""
        if self.token_refresh_callback is None:
            return False
        
        try:
            new_token = self.token_refresh_callback()
            if new_token:
                self.access_token = new_token
                self.stats['token_refreshes'] += 1
                return True
        except Exception as e:
            print(f"⚠️ Token refresh failed: {e}")
        
        return False
    
    def _make_api_call_with_retry(self, url: str, logger=None) -> Optional[dict]:
        """Make API call with exponential backoff retry logic"""
        self.stats['api_calls'] += 1
        retry_count = 0
        
        while retry_count <= self.max_retries:
            try:
                response = requests.get(url, headers=self._get_headers(), timeout=30)
                
                if response.status_code == 200:
                    return response.json()
                
                elif response.status_code == 401:
                    if logger:
                        logger.warning(f"Token expired (401), attempting refresh...")
                    if self._refresh_token():
                        if logger:
                            logger.info("✓ Token refreshed, retrying request")
                        continue
                    else:
                        if logger:
                            logger.error("Failed to refresh token, cannot continue")
                        raise Exception("Token expired and refresh failed")
                
                elif response.status_code == 429:
                    self.stats['rate_limit_429'] += 1
                    retry_after = int(response.headers.get('Retry-After', self.base_delay * (2 ** retry_count)))
                    wait_time = min(retry_after, self.max_delay)
                    
                    if logger:
                        logger.warning(f"Rate limited (429), waiting {wait_time}s before retry {retry_count + 1}/{self.max_retries}")
                    
                    time.sleep(wait_time)
                    retry_count += 1
                
                elif response.status_code in [503, 504]:
                    wait_time = min(self.base_delay * (2 ** retry_count), self.max_delay)
                    
                    if logger:
                        logger.warning(f"Service error ({response.status_code}), waiting {wait_time}s before retry {retry_count + 1}/{self.max_retries}")
                    
                    time.sleep(wait_time)
                    retry_count += 1
                
                elif response.status_code == 404:
                    if logger:
                        logger.warning(f"Group not found (404): {url}")
                    return None
                
                elif response.status_code == 403:
                    if logger:
                        logger.warning(f"Access denied (403): {url}")
                    return None
                
                else:
                    if logger:
                        logger.error(f"API error {response.status_code}: {response.text}")
                    return None
            
            except requests.exceptions.Timeout:
                wait_time = min(self.base_delay * (2 ** retry_count), self.max_delay)
                if logger:
                    logger.warning(f"Request timeout, waiting {wait_time}s before retry {retry_count + 1}/{self.max_retries}")
                time.sleep(wait_time)
                retry_count += 1
            
            except requests.exceptions.RequestException as e:
                if logger:
                    logger.error(f"Request exception: {e}")
                return None
        
        if logger:
            logger.error(f"Max retries ({self.max_retries}) exceeded for: {url}")
        return None
    
    def _handle_pagination_for_group(
        self,
        group_id: str,
        initial_members: List[Dict],
        next_link: Optional[str],
        logger=None
    ) -> List[Dict]:
        """Handle pagination for a single group after initial batch fetch"""
        all_members = initial_members.copy()
        
        if not next_link:
            return all_members
        
        if logger:
            logger.debug(f"  → Group {group_id} has pagination, fetching additional pages...")
        
        url = next_link
        while url:
            self.stats['pagination_calls'] += 1
            response_data = self._make_api_call_with_retry(url, logger)
            
            if not response_data:
                break
            
            members = response_data.get('value', [])
            all_members.extend(members)
            
            url = response_data.get('@odata.nextLink')
            
            if url:
                time.sleep(self.batch_delay)
        
        if logger:
            logger.debug(f"  ✓ Group {group_id}: {len(all_members)} total members (after pagination)")
        
        return all_members
    
    def _make_batch_request(
        self,
        group_ids: List[str],
        logger=None
    ) -> Dict[str, Optional[List[Dict]]]:
        """Make a JSON batch request with pagination support"""
        if len(group_ids) > 20:
            if logger:
                logger.warning(f"Batch size {len(group_ids)} exceeds limit of 20, truncating")
            group_ids = group_ids[:20]
        
        batch_requests = []
        for idx, group_id in enumerate(group_ids):
            batch_requests.append({
                "id": str(idx),
                "method": "GET",
                "url": f"/groups/{group_id}/members?$select=id,userPrincipalName,displayName"
            })
        
        batch_payload = {"requests": batch_requests}
        
        self.stats['batch_requests'] += 1
        self.stats['batched_api_calls'] += len(group_ids)
        
        if logger:
            logger.debug(f"Making batch request for {len(group_ids)} groups")
        
        try:
            response = requests.post(
                "https://graph.microsoft.com/v1.0/$batch",
                headers=self._get_headers(),
                json=batch_payload,
                timeout=60
            )
            
            if response.status_code != 200:
                if logger:
                    logger.error(f"Batch request failed: {response.status_code} - {response.text}")
                return {}
            
            batch_response = response.json()
            
            results = {}
            for response_item in batch_response.get('responses', []):
                request_id = int(response_item.get('id'))
                group_id = group_ids[request_id]
                status = response_item.get('status')
                
                if status == 200:
                    body = response_item.get('body', {})
                    initial_members = body.get('value', [])
                    next_link = body.get('@odata.nextLink')
                    
                    if next_link:
                        all_members = self._handle_pagination_for_group(
                            group_id,
                            initial_members,
                            next_link,
                            logger
                        )
                        results[group_id] = all_members
                    else:
                        results[group_id] = initial_members
                    
                    if logger:
                        logger.debug(f"  ✓ Group {group_id}: {len(results[group_id])} members")
                else:
                    if logger:
                        logger.warning(f"  ✗ Group {group_id}: status {status}")
                    results[group_id] = None
            
            return results
            
        except requests.exceptions.RequestException as e:
            if logger:
                logger.error(f"Batch request exception: {e}")
            return {}
    
    def _get_group_members_paginated(self, group_id: str, logger=None) -> List[Dict]:
        """Get all members of a group with pagination support"""
        if self.use_global_cache and group_id in self.global_cache:
            self.stats['global_cache_hits'] += 1
            if logger:
                logger.debug(f"GLOBAL cache hit for group: {group_id}")
            return self.global_cache[group_id]
        
        all_members = []
        url = f"https://graph.microsoft.com/v1.0/groups/{group_id}/members?$select=id,userPrincipalName,displayName"
        
        while url:
            response_data = self._make_api_call_with_retry(url, logger)
            
            if not response_data:
                break
            
            members = response_data.get('value', [])
            all_members.extend(members)
            
            url = response_data.get('@odata.nextLink')
            if url:
                time.sleep(self.batch_delay)
        
        if self.use_global_cache:
            self.global_cache[group_id] = all_members
            if logger:
                logger.debug(f"Stored group {group_id} in GLOBAL cache ({len(all_members)} members)")
        
        return all_members
    
    def _get_multiple_groups_batched(
        self,
        group_ids: List[str],
        logger=None
    ) -> Dict[str, List[Dict]]:
        """Fetch multiple groups using JSON batching with pagination support"""
        results = {}
        groups_to_fetch = []
        
        for group_id in group_ids:
            if self.use_global_cache and group_id in self.global_cache:
                self.stats['global_cache_hits'] += 1
                results[group_id] = self.global_cache[group_id]
                if logger:
                    logger.debug(f"GLOBAL cache hit for group: {group_id}")
            else:
                groups_to_fetch.append(group_id)
        
        if groups_to_fetch:
            for i in range(0, len(groups_to_fetch), self.batch_size):
                batch = groups_to_fetch[i:i + self.batch_size]
                
                batch_results = self._make_batch_request(batch, logger)
                
                for group_id, members in batch_results.items():
                    if members is not None:
                        if self.use_global_cache:
                            self.global_cache[group_id] = members
                        results[group_id] = members
                
                if i + self.batch_size < len(groups_to_fetch):
                    time.sleep(self.batch_delay)
        
        return results
    
    def expand_group_iterative(
        self,
        root_group_id: str,
        root_group_name: str,
        logger=None
    ) -> List[Dict[str, str]]:
        """ITERATIVE group expansion using a work queue"""
        memberships = []
        local_cache: Set[str] = set()
        work_queue = deque([(root_group_id, root_group_name, root_group_name, 0)])
        
        if logger:
            cache_type = "GLOBAL" if self.use_global_cache else "LOCAL"
            batch_mode = "BATCHING" if self.use_json_batching else "SEQUENTIAL"
            logger.info(f"Starting expansion: {root_group_name} [Cache: {cache_type}, Mode: {batch_mode}]")
        
        while len(work_queue) > 0:
            
            if self.use_json_batching:
                groups_to_process = []
                
                while len(groups_to_process) < self.batch_size and len(work_queue) > 0:
                    group_info = work_queue.popleft()
                    group_id = group_info[0]
                    
                    if group_id not in local_cache:
                        groups_to_process.append(group_info)
                        local_cache.add(group_id)
                
                if not groups_to_process:
                    continue
                
                group_ids_to_fetch = [g[0] for g in groups_to_process]
                batch_results = self._get_multiple_groups_batched(group_ids_to_fetch, logger)
                
                for group_id, group_name, current_path, nesting_level in groups_to_process:
                    members = batch_results.get(group_id, [])
                    
                    if nesting_level > self.stats['max_nesting_depth']:
                        self.stats['max_nesting_depth'] = nesting_level
                    
                    self._process_members(
                        members,
                        root_group_id,
                        root_group_name,
                        current_path,
                        nesting_level,
                        work_queue,
                        memberships
                    )
            
            else:
                current_group_id, current_group_name, current_path, nesting_level = work_queue.popleft()
                
                if nesting_level > self.stats['max_nesting_depth']:
                    self.stats['max_nesting_depth'] = nesting_level
                
                if current_group_id in local_cache:
                    self.stats['local_cache_hits'] += 1
                    if logger:
                        logger.debug(f"LOCAL cache hit: {current_group_name}")
                    continue
                
                local_cache.add(current_group_id)
                members = self._get_group_members_paginated(current_group_id, logger)
                
                self._process_members(
                    members,
                    root_group_id,
                    root_group_name,
                    current_path,
                    nesting_level,
                    work_queue,
                    memberships
                )
        
        if logger:
            logger.info(f"✓ Completed expansion: {root_group_name} ({len(memberships)} memberships)")
        
        return memberships
    
    def _process_members(
        self,
        members: List[Dict],
        root_group_id: str,
        root_group_name: str,
        current_path: str,
        nesting_level: int,
        work_queue: deque,
        memberships: List[Dict]
    ):
        """Helper method to process members"""
        for member in members:
            member_type = member.get('@odata.type', '')
            member_id = member.get('id', 'unknown')
            member_display_name = member.get('displayName', 'Unknown')
            
            if member_type == '#microsoft.graph.user':
                user_principal_name = member.get('userPrincipalName', 'unknown')
                
                memberships.append({
                    'group_id': root_group_id,
                    'group_name': root_group_name,
                    'user_id': member_id,
                    'user_principal_name': user_principal_name,
                    'user_display_name': member_display_name,
                    'membership_path': f"{current_path}|{member_display_name}",
                    'nesting_level': nesting_level + 1
                })
            
            elif member_type == '#microsoft.graph.group':
                nested_group_name = member_display_name
                nested_path = f"{current_path}|{nested_group_name}"
                
                work_queue.append((
                    member_id,
                    nested_group_name,
                    nested_path,
                    nesting_level + 1
                ))
    
    def write_to_lakehouse_batched(
        self,
        group_ids: Dict[str, str],
        table_name: str,
        spark,
        write_batch_size: int = 500,
        clear_table: bool = True,  # NEW: Option to clear table before writing
        logger=None
    ) -> int:
        """
        Expand groups and write to Delta Lake in batches.
        
        NEW: clear_table parameter controls whether to overwrite existing data
        - clear_table=True: Clears table on first write (prevents duplicates)
        - clear_table=False: Appends to existing data
        """
        from pyspark.sql.types import StructType, StructField, StringType, IntegerType
        
        all_memberships = []
        total_written = 0
        groups_processed = 0
        first_write = True  # Track if this is the first write
        
        if logger:
            cache_type = "GLOBAL" if self.use_global_cache else "LOCAL"
            batch_mode = "JSON-BATCHING" if self.use_json_batching else "SEQUENTIAL"
            logger.info(f"Starting expansion for {len(group_ids)} groups")
            logger.info(f"  Cache: {cache_type}, Mode: {batch_mode}, Write batch: {write_batch_size}")
            logger.info(f"  Clear table: {clear_table}")
        
        # Define schema
        schema = StructType([
            StructField("group_id", StringType(), True),
            StructField("group_name", StringType(), True),
            StructField("user_id", StringType(), True),
            StructField("user_principal_name", StringType(), True),
            StructField("user_display_name", StringType(), True),
            StructField("membership_path", StringType(), True),
            StructField("nesting_level", IntegerType(), True)
        ])
        
        # Check if table exists
        table_exists = False
        try:
            spark.sql(f"DESCRIBE TABLE {table_name}")
            table_exists = True
            if logger:
                logger.info(f"✓ Table '{table_name}' exists")
        except Exception:
            if logger:
                logger.info(f"⚠ Table '{table_name}' does not exist, will create")
        
        # Process groups
        for group_id, group_name in group_ids.items():
            groups_processed += 1
            
            if logger and groups_processed % 10 == 0:
                logger.info(f"Progress: {groups_processed}/{len(group_ids)} groups")
            
            # Expand this group
            memberships = self.expand_group_iterative(group_id, group_name, logger)
            all_memberships.extend(memberships)
            
            # Write batch if threshold reached
            if len(all_memberships) >= write_batch_size:
                if logger:
                    logger.info(f"Writing batch of {len(all_memberships)} records...")
                
                df = spark.createDataFrame(all_memberships)
                
                # First write: use overwrite or append based on clear_table parameter
                if first_write:
                    if clear_table:
                        # OVERWRITE on first write (clears existing data)
                        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
                        if logger:
                            logger.info(f"✓ First batch written (table cleared)")
                    else:
                        # APPEND on first write (keeps existing data)
                        df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(table_name)
                        if logger:
                            logger.info(f"✓ First batch written (appended to existing)")
                    first_write = False
                else:
                    # All subsequent writes: always append
                    df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(table_name)
                
                total_written += len(all_memberships)
                all_memberships = []
                
                if logger:
                    logger.info(f"✓ Batch written. Total: {total_written}")
                
                time.sleep(self.batch_delay)
        
        # Write remaining records
        if all_memberships:
            if logger:
                logger.info(f"Writing final batch of {len(all_memberships)} records...")
            
            df = spark.createDataFrame(all_memberships)
            
            # Handle final batch
            if first_write:
                if clear_table:
                    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
                    if logger:
                        logger.info(f"✓ Final batch written (table cleared)")
                else:
                    df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(table_name)
                    if logger:
                        logger.info(f"✓ Final batch written (appended to existing)")
            else:
                df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(table_name)
            
            total_written += len(all_memberships)
        
        # Print statistics
        if logger:
            logger.info(f"\n{'='*70}")
            logger.info(f"PROCESSING COMPLETE")
            logger.info(f"{'='*70}")
            logger.info(f"Total memberships written: {total_written}")
            logger.info(f"\nAPI Statistics:")
            logger.info(f"  Individual API calls: {self.stats['api_calls']}")
            logger.info(f"  Pagination calls: {self.stats['pagination_calls']}")
            
            if self.use_json_batching:
                logger.info(f"  Batch requests: {self.stats['batch_requests']}")
                logger.info(f"  Batched API calls: {self.stats['batched_api_calls']}")
                total_calls = self.stats['api_calls'] + self.stats['batch_requests']
                saved_calls = self.stats['batched_api_calls'] - self.stats['batch_requests']
                logger.info(f"  Total HTTP requests: {total_calls}")
                logger.info(f"  Requests saved by batching: {saved_calls}")
            
            if self.use_global_cache:
                logger.info(f"\nCache Statistics:")
                logger.info(f"  Groups in cache: {len(self.global_cache)}")
                logger.info(f"  Global cache hits: {self.stats['global_cache_hits']}")
                logger.info(f"  Local cache hits: {self.stats['local_cache_hits']}")
            
            logger.info(f"\nOther Statistics:")
            logger.info(f"  Token refreshes: {self.stats['token_refreshes']}")
            logger.info(f"  Rate limit hits: {self.stats['rate_limit_429']}")
            logger.info(f"  Max nesting depth: {self.stats['max_nesting_depth']}")
            logger.info(f"{'='*70}\n")
        
        return total_written

print("✓ EntraIDGroupExpander class loaded (FINAL VERSION)")
print("  Key features:")
print("  - Iterative processing with work queue")
print("  - Optional GLOBAL cache across all root groups")
print("  - Optional JSON batching with proper pagination")
print("  - NEW: clear_table parameter to prevent duplicates")
print("  - Automatic token refresh and retry logic")
print("\n  Usage:")
print("    expander = EntraIDGroupExpander(token, use_global_cache=True, use_json_batching=True)")
print("    expander.write_to_lakehouse_batched(..., clear_table=True)  # Clears old data")
print("    expander.write_to_lakehouse_batched(..., clear_table=False) # Keeps old data")


# Sec 03 - Create an MSAL Application (Microsoft Authentication Library app)

In [ ]:
# Create a Confidential Client Application
app = msal.ConfidentialClientApplication(
    client_id=client_id,
    client_credential=client_secret,
    authority=f"https://login.microsoftonline.com/{tenant_id}"
)

# Sec 04 - Get Groups to Expand
The initial list of groups that will be expanded

In [ ]:
df_groups = spark.sql("""
    SELECT DISTINCT GraphId 
    FROM FUAM_Lakehouse.workspaces_scanned_users 
    WHERE PrincipalType = 'Group'
    AND GraphId IS NOT NULL
""")
# df_groups = spark.sql("""
#     SELECT DISTINCT GraphId 
#     FROM FUAM_Lakehouse.workspaces_scanned_users 
#     WHERE PrincipalType = 'Group'
#     AND GraphId = 'dcd70cc4-71d3-46fb-98aa-6f67891237fc'
# """)
#display(df_groups)

# Sec 05 - Dataframe to List of Dicts, and reading the displayName of the group

In [ ]:
# ============================================================================
# Fetch displayNames from Graph API for all groups using GroupMetadataFetcher
# ============================================================================

# Get initial token
print("Fetching access token...")
token_response = app.acquire_token_for_client(scopes=["https://graph.microsoft.com/.default"])
if "access_token" not in token_response:
    raise Exception(f"Failed to acquire token: {token_response.get('error_description', 'Unknown error')}")
access_token = token_response["access_token"]
print("✓ Access token acquired")

# Define token refresh callback
def get_fresh_token():
    """Function to get a fresh access token - called automatically when token expires"""
    print("Refreshing access token...")
    token_response = app.acquire_token_for_client(scopes=["https://graph.microsoft.com/.default"])
    if "access_token" in token_response:
        print("✓ Token refreshed successfully")
        return token_response["access_token"]
    return None

# Initialize metadata fetcher
fetcher = GroupMetadataFetcher(
    access_token=access_token,
    token_refresh_callback=get_fresh_token,
    max_retries=3,
    batch_size=20,
    batch_delay=0.2,
    use_cache=True  # Cache results to avoid duplicate API calls
)

# Get list of group IDs from dataframe
group_ids_list = [row['GraphId'] for row in df_groups.collect()]
print(f"\nFetching displayNames for {len(group_ids_list)} groups using batched API calls...")

# Fetch displayNames using the class
groups_to_expand = fetcher.fetch_display_names(group_ids_list)

print(f"✓ Successfully fetched {len(groups_to_expand)} group displayNames")
print(f"\nSample of fetched displayNames:")
for i, (group_id, display_name) in enumerate(list(groups_to_expand.items())[:5]):
    print(f"  {group_id} -> {display_name}")

# Print statistics
fetcher.print_stats()

# Sec 06 - Executor

In [ ]:
# ============================================================================
# CELL 7 - EXECUTOR (WITH FRESH TABLE NAME)
# ============================================================================
# Initialize CSV logger
with FabricLoggerCSV(
    "group_expansion", 
    lakehouse_path="/lakehouse/default/Files/logs",
    enable_detail_csv=ENABLE_DETAIL_CSV,   # Toggle detailed CSV logging
    enable_summary_csv=ENABLE_SUMMARY_CSV  # Toggle summary CSV
) as logger:
    
    start_time = datetime.now()
    logger.info(f"Starting group expansion process (ITERATIVE VERSION)")
    logger.info(f"Total groups to process: {len(groups_to_expand)}")
    logger.info(f"Write batch size: 500")
    logger.info(f"Table name: {TABLE_NAME}")
    
    # Record initial metrics in summary CSV
    logger.set_summary_metric('total_groups', len(groups_to_expand))
    logger.set_summary_metric('batch_size', 500)
    logger.set_summary_metric('expansion_method', 'iterative')
    logger.set_summary_metric('table_name', TABLE_NAME)
    
    def get_fresh_token():
        """Function to get a fresh access token - called automatically when token expires"""
        logger.info("Refreshing access token...")
        token_response = app.acquire_token_for_client(scopes=["https://graph.microsoft.com/.default"])
        if "access_token" in token_response:
            access_token = token_response["access_token"]
            logger.info("✓ Token refreshed successfully")
        return access_token

    # Get initial token
    access_token = get_fresh_token()
    logger.info("✓ Initial token acquired")

    # Initialize expander with automatic token refresh
    expander = EntraIDGroupExpander(
        access_token=access_token,
        token_refresh_callback=get_fresh_token,
        max_retries=5,
        base_delay=2.0,
        max_delay=120.0,
        batch_delay=0.2,
        use_global_cache=True,    # ✅ Prevents duplicate fetches
        use_json_batching=True,   # ✅ Batches up to 20 requests
    )
    logger.info("✓ EntraIDGroupExpander initialized (iterative mode)")

    # Expand groups and write to Delta table in batches
    logger.info("Starting group expansion and Delta writes...")
    total_records = expander.write_to_lakehouse_batched(
        group_ids=groups_to_expand,
        table_name=TABLE_NAME,  # NEW TABLE NAME
        spark=spark,
        write_batch_size= WRITE_DELTA_TABLE_BATCH_SIZE,
        logger=logger
    )

    # Calculate final statistics
    end_time = datetime.now()
    elapsed = (end_time - start_time).total_seconds()
    
    # Record metrics in summary CSV
    logger.set_summary_metric('total_records', total_records)
    logger.set_summary_metric('groups_per_second', round(len(groups_to_expand)/elapsed, 2))
    logger.set_summary_metric('api_calls', expander.stats.get('api_calls', 0))
    logger.set_summary_metric('cache_hits', expander.stats.get('cache_hits', 0))
    logger.set_summary_metric('token_refreshes', expander.stats.get('token_refreshes', 0))
    logger.set_summary_metric('max_nesting_depth', expander.stats.get('max_nesting_depth', 0))
    
    logger.info("=== Final Summary ===")
    logger.info(f"Total groups processed: {len(groups_to_expand)}")
    logger.info(f"Total memberships created: {total_records:,}")
    logger.info(f"Max nesting depth reached: {expander.stats.get('max_nesting_depth', 0)}")
    logger.info(f"Elapsed time: {elapsed:.2f} seconds")
    logger.info(f"Processing rate: {len(groups_to_expand)/elapsed:.1f} groups/second")
    logger.info(f"API calls made: {expander.stats.get('api_calls', 0)}")
    logger.info(f"Cache hits: {expander.stats.get('cache_hits', 0)}")
    logger.info(f"Token refreshes: {expander.stats.get('token_refreshes', 0)}")
    
    # Minimal console output (doesn't bloat notebook)
    print(f"\n{'='*70}")
    print(f"✓ Complete! Processed {len(groups_to_expand)} groups in {elapsed:.1f}s")
    print(f"✓ Total memberships: {total_records:,}")
    print(f"✓ Rate: {len(groups_to_expand)/elapsed:.1f} groups/second")
    print(f"✓ Max nesting depth: {expander.stats.get('max_nesting_depth', 0)}")
    print(f"✓ Table: {TABLE_NAME}")
    print(f"{'='*70}")
    
    # Show what was created
    if ENABLE_DETAIL_CSV:
        print(f"\n📋 Detail CSV: {logger.get_detail_path()}")
        print(f"   Click to view in Fabric UI - easy to read!")
    if ENABLE_SUMMARY_CSV:
        print(f"📊 Summary CSV: {logger.get_summary_path()}")
    
    if not ENABLE_DETAIL_CSV and not ENABLE_SUMMARY_CSV:
        print(f"\n💡 Both logging options disabled")
    else:
        print(f"\n💡 View CSVs: Lakehouse → Files → logs folder")
        print(f"💡 Query in Spark:")
        if ENABLE_DETAIL_CSV:
            print(f"   Detail: spark.read.csv('{logger.get_detail_path()}', header=True)")
        if ENABLE_SUMMARY_CSV:
            print(f"   Summary: spark.read.csv('{logger.get_summary_path()}', header=True)")

# Summary statistics
print(f"\n{'='*70}")
print(f"Summary statistics from table: {TABLE_NAME}")
print(f"{'='*70}")
spark.sql(f"""
    SELECT 
        group_name,
        COUNT(DISTINCT user_principal_name) as unique_users,
        COUNT(*) as total_memberships,
        MAX(nesting_level) as max_nesting_level
    FROM {TABLE_NAME}
    GROUP BY group_name
    ORDER BY unique_users DESC
    LIMIT 50
""").show(truncate=False)

# Sec 07 - Join the dataframe with the FUAM table 'workspaces_scanned_users' and write the table to the lakehouse

In [ ]:
# The FUAM Table
df_workspaces_scanned_users = spark.sql("SELECT * FROM FUAM_Lakehouse.workspaces_scanned_users")
# The table containing the expanded users
df_expanded_groups = spark.sql(f"""
SELECT * FROM
{TABLE_NAME}
""")


df_join = df_workspaces_scanned_users.join(df_expanded_groups, df_workspaces_scanned_users.GraphId == df_expanded_groups.group_id, "leftouter")
df_join = df_join \
    .withColumn("user_principal_name_expanded", when(df_join.PrincipalType == 'User', df_join.Identifier).otherwise(df_join.user_principal_name))
#display(df_join)

# Writing the joined table to the lakehouse
df_join.write \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(f"{TABLE_NAME_FOR_RLS}")

df_join.write \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(f"{TABLE_NAME_FOR_RLS}")

# Debugging
This is for debugging single groups, assuming Cell 1 to Cell 4 have been run in the current session context.

In [ ]:
# def get_fresh_token():
#         """Function to get a fresh access token - called automatically when token expires"""
#         #logger.info("Refreshing access token...")
#         token_response = app.acquire_token_for_client(scopes=["https://graph.microsoft.com/.default"])
#         if "access_token" in token_response:
#             access_token = token_response["access_token"]
#             #logger.info("✓ Token refreshed successfully")
#         return access_token


# access_token = get_fresh_token ()

# expander = EntraIDGroupExpander(
#         access_token=access_token,
#         token_refresh_callback=get_fresh_token,
#         max_retries=5,
#         base_delay=2.0,
#         max_delay=120.0,
#         batch_delay=0.2,
#         use_global_cache=True,    # ✅ Prevents duplicate fetches
#         use_json_batching=True,   # ✅ Batches up to 20 requests
#     )

# thisorthat = expander.expand_group_iterative('a group id', 'the same group id')
# print(len(thisorthat))